# Planner Tests

Most data from lecture_examples/IP-X-1-Automated_PlanerTest.ipynb

Adding all planners
===========


In [ ]:
import sys
from pathlib import Path
import matplotlib.pylab as plt
%matplotlib inline
import traceback

# Define paths
REPO_ROOT = Path.cwd().parent               # Repo root (ikp-sbl)
LECTURE_EXAMPLES = REPO_ROOT / "lecture_examples"

# Add BOTH to sys.path
for path in [str(REPO_ROOT), str(LECTURE_EXAMPLES)]:
    if path not in sys.path:
        sys.path.insert(0, path)

# 1. Imports from lecture_examples
from IPPerfMonitor import IPPerfMonitor
import IPBasicPRM
import IPVISBasicPRM
import IPLazyPRM
import IPVISLazyPRM
import IPRRT
import IPVISRRT

# 2. Other imports
from planners.SBL import BidirectionalSBL
from modules import IPVISsbl
# from modules import draw

Set-up of the test scenario and the configuration for all planners
===================================

Following a procedure to compare all discussed planners are shown:

1. Configuration for every planner is defined
2. The configuration and the planner is stored in the variable setup, a Python dict()
3. The variable setup is then used to uniformly execute the planners


In [ ]:
# Create a factory for your parameter variations
plannerFactory = dict()

config = BidirectionalSBL.DEFAULT_CONFIG.copy()
config["eta"] = 2.0
plannerFactory[f"SBL"] = [BidirectionalSBL, config, IPVISsbl.sblVisualize]

# Example: Testing different Eta settings
# for eta in [0.5, 1.0, 2.0]:
#     config = BidirectionalSBL.DEFAULT_CONFIG.copy()
#     config["eta"] = eta
#     plannerFactory[f"SBL_eta_{eta}"] = [BidirectionalSBL, config]

# # Example: Testing different Goal Bias settings
# for bias in [0.2, 0.5, 0.8]:
#     config = BidirectionalSBL.DEFAULT_CONFIG.copy()
#     config["goal_bias"] = bias
#     plannerFactory[f"SBL_bias_{bias}"] = [BidirectionalSBL, config]

In [ ]:
basicConfig = dict()
basicConfig["radius"] = 3
basicConfig["numNodes"] = 200
plannerFactory["basePRM"] = [IPBasicPRM.BasicPRM, basicConfig, IPVISBasicPRM.basicPRMVisualize]

lazyConfig = dict()
lazyConfig["initialRoadmapSize"] = 10
lazyConfig["updateRoadmapSize"]  = 5 
lazyConfig["kNearest"] = 8
lazyConfig["maxIterations"] = 20
plannerFactory["lazyPRM"] = [IPLazyPRM.LazyPRM, lazyConfig, IPVISLazyPRM.lazyPRMVisualize]

rrtSimpleConfig = dict()
rrtSimpleConfig["numberOfGeneratedNodes"] = 100 
rrtSimpleConfig["testGoalAfterNumberOfNodes"]  = 10
plannerFactory["simpleRRT"] = [IPRRT.RRTSimple, rrtSimpleConfig, IPVISRRT.rrtPRMVisualize]




In [ ]:
class ResultCollection (object):
    
    def __init__(self, plannerFactoryName, planner, benchmark, solution, perfDataFrame):
        self.plannerFactoryName = plannerFactoryName
        self.planner = planner
        self.benchmark = benchmark
        self.solution = solution
        self.perfDataFrame = perfDataFrame

In [ ]:
import lecture_examples.IPTestSuite as ts
from shapely.geometry import Point, Polygon, LineString
from shapely import plotting


In [ ]:
fullBenchList = ts.benchList

for benchmark in fullBenchList:
    print(benchmark.name)

In [ ]:
for benchmark in fullBenchList:
    fig_local = plt.figure(figsize=(7,7))
    ax = fig_local.add_subplot(1,1,1)
    title = benchmark.name
    ax.set_title(title)
    
    limits = benchmark.collisionChecker.getEnvironmentLimits()
    
    # Add a visual margin so the buffered points are not cropped at the map edges
    margin = 1.0 
    ax.set_xlim(limits[0][0] - margin, limits[0][1] + margin)
    ax.set_ylim(limits[1][0] - margin, limits[1][1] + margin)
    plotting.plot_points(Point(benchmark.startList[0]).buffer(.3), color="g", ax=ax, label="Start")
    plotting.plot_points(Point(benchmark.goalList[0]).buffer(.3), color="b", ax=ax, label="Goal")
    
    benchmark.collisionChecker.drawObstacles(ax)

    # Optional: add legend to easily identify start/goal
    ax.legend(loc="upper right")

In [ ]:
resultList = list()
testList = fullBenchList

for key, producer in list(plannerFactory.items()):
    print(key, producer)

    planner_class = producer[0]
    config = producer[1]

    for benchmark in testList:
        print ("Planning: " + key + " - " + benchmark.name)
        # 1. Clear data from previous runs
        IPPerfMonitor.clearData()

        # 2. Instantiate and run
        # Note: Depending on the constructor, some planners might not accept config here. 
        # But based on our factory setup, we pass it down if they accept it.
        try:
            planner = planner_class(benchmark.collisionChecker, config=config)
        except TypeError:
            # Fallback if lecture planners don't accept config in their constructor
            planner = planner_class(benchmark.collisionChecker)
        
        try:
            path = planner.planPath(
                benchmark.startList, 
                benchmark.goalList, 
                config
            )

            # 4. Store result using the ResultCollection class
            resultList.append(
                ResultCollection(
                    key,
                    planner, 
                    benchmark, 
                    path, 
                    IPPerfMonitor.dataFrame()
                )
            )
            
        except Exception as e:
            # THIS WILL REVEAL THE EXACT CRASH!
            print(f"CRASH DETECTED for {key} on {benchmark.name}:")
            traceback.print_exc()
            print("-" * 40)


In [ ]:
import matplotlib.pyplot as plt

for result in resultList:
    
    fig_local = plt.figure(figsize=(10,10))
    ax = fig_local.add_subplot(1,1,1)
    title = result.plannerFactoryName + " - " + result.benchmark.name
    if result.solution == []:
        title += " (No path found!)"
    title += "\n Assumed complexity level " + str(result.benchmark.level)
    ax.set_title(title)
    try:
        plannerFactory[result.plannerFactoryName][2](result.planner, result.solution, ax=ax, nodeSize=100)
    except Exception as e:
        print ("Error", e)
        pass
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for bench in testList:
    title = bench.name
    pathLength = dict()
    planningTime = dict()
    roadmapSize  = dict()
    
    for result in resultList:
        if result.benchmark.name == bench.name:
            # 1. Path length
            pathLength[result.plannerFactoryName] = len(result.solution) if result.solution else 0
            
            # 2. Planning Time
            try:
                # Try getting it from the PerfMonitor DataFrame (works for lecture examples)
                planningTime[result.plannerFactoryName] = result.perfDataFrame.groupby(["name"]).sum(numeric_only=True)["time"]["planPath"]
            except KeyError:
                # Fallback: SBL module might not sync perfectly with the notebook's IPPerfMonitor
                if hasattr(result.planner, 'stats') and hasattr(result.planner.stats, 'planning_time'):
                    # Use SBL's internal PlannerStats tracker
                    planningTime[result.plannerFactoryName] = result.planner.stats.planning_time
                else:
                    planningTime[result.plannerFactoryName] = 0.0

            # 3. Roadmap Size
            if hasattr(result.planner, 'stats') and hasattr(result.planner.stats, 'total_nodes_start_tree'):
                # For SBL: Use the exact node counts tracked in PlannerStats
                roadmapSize[result.plannerFactoryName] = result.planner.stats.total_nodes_start_tree + result.planner.stats.total_nodes_goal_tree
            elif hasattr(result.planner, 'graph'):
                # For standard PRM planners
                roadmapSize[result.plannerFactoryName] = result.planner.graph.size()
            else:
                roadmapSize[result.plannerFactoryName] = 0

    # Skip plotting if no results were found for this benchmark
    if not pathLength:
        continue

    fig, ax = plt.subplots(figsize=(10, 6)) # Made slightly wider to fit text labels

    ax.set_title(f"Benchmark: {title}", fontsize=14, fontweight='bold', pad=25)

    width = 0.2
    x_indices = np.arange(len(pathLength.keys()))

    # --- 1. Path Length (Blue) ---
    bar1 = ax.bar(x_indices, pathLength.values(), width, color="blue")
    ax.set_ylabel("Number of nodes in path", color="blue")
    ax.set_xticks(x_indices + width)
    ax.set_xticklabels(pathLength.keys())
    
    # Add labels above blue bars (format as integer)
    ax.bar_label(bar1, padding=3, fmt='%d', color="blue")

    # --- 2. Planning Time (Red/Yellow) ---
    ax2 = ax.twinx()
    bar2 = ax2.bar(x_indices + width, planningTime.values(), width, color="red")
    ax2.set_ylabel("Planning time (s)", color="orange")

    # Add coloring and patterns on axis two for failures
    hatches = ['x' if length==0 else '' for length in pathLength.values()]
    color   = ['red' if length==0 else 'orange' for length in pathLength.values()]
    for i, thisbar in enumerate(bar2.patches):
        thisbar.set_facecolor(color[i])
        thisbar.set_hatch(hatches[i])
        
    # Add labels above time bars (format as float with 3 decimal places)
    # Using 'black' for visibility against both orange and red
    ax2.bar_label(bar2, padding=3, fmt='%.2f', color="black")

    # --- 3. Roadmap Size (Purple) ---
    ax3 = ax.twinx()
    bar3 = ax3.bar(x_indices + 2*width, roadmapSize.values(), width, color="purple")
    ax3.set_ylabel("Roadmap size", color="purple")
    ax3.spines['right'].set_position(('axes', 1.15))
    ax3.spines['right'].set_color("purple")
    
    # Add labels above purple bars (format as integer)
    ax3.bar_label(bar3, padding=3, fmt='%d', color="purple")

    # Clean up layout
    plt.tight_layout()
    plt.show()